# 02 — The residual was the gradient

*Chapter 08 · Gradient Boosting · Notebook 2 of 6*

In Notebook 1 we built gradient boosting without ever using the word "gradient." We fit the residuals,
added a shrunken slice, repeated — and matched the library to machine precision. This notebook reveals
what was really happening: **the residual we fit is the negative gradient of a loss**, and the whole
loop is **gradient descent — carried out in a space of functions.**

**Prerequisites.** Notebook 1 (the residual-fitting loop and its exact parity with
`GradientBoostingRegressor`); Chapter 03, Notebook 4 (gradient descent — writing a loss and stepping
downhill); Module 00 (mean squared error).

**What you'll be able to do.**
- Show that the residual `y − F` is the negative gradient of the squared-error loss.
- Explain boosting as gradient descent in function space.
- Watch the loss descend, and see the path on a 2-D slice.
- Explain why a different loss changes what each tree fits — the key to the whole family.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error
from sklearn.tree import DecisionTreeRegressor

from ml_course import viz
from ml_course.colors import COLORS

viz.use_course_style()
SEED, NU, MAX_DEPTH, N_TREES = 0, 0.3, 2, 100

# The same 1-D problem and by-hand loop as Notebook 1 — re-examined here.
rng = np.random.default_rng(SEED)
n = 120
x = np.sort(rng.uniform(0.0, 2.0 * np.pi, n))
y = np.sin(x) + rng.normal(0.0, 0.25, n)
X = x.reshape(-1, 1)

F = np.full(n, y.mean())
trees, stages = [], [F.copy()]
loss_curve = [0.5 * np.sum((y - F) ** 2)]
for _ in range(N_TREES):
    residual = y - F
    tree = DecisionTreeRegressor(max_depth=MAX_DEPTH, random_state=SEED).fit(X, residual)
    F = F + NU * tree.predict(X)
    trees.append(tree)
    stages.append(F.copy())
    loss_curve.append(0.5 * np.sum((y - F) ** 2))
loss_curve = np.array(loss_curve)
print(f"rebuilt Notebook 1's model: {N_TREES} trees, final train MSE = {mean_squared_error(y, F):.4f}")

## A recap, and a question

Notebook 1's recipe: start at the mean, compute the **residuals** `y − F`, fit a shallow tree to them,
add a fraction `ν`, repeat. It worked, and it equalled `GradientBoostingRegressor` exactly.

Now recall how we trained logistic regression in Chapter 03, Notebook 4: we wrote the loss as a function
of the **weights**, computed its **gradient**, and stepped downhill — gradient descent. That is the
standard way to minimize a loss.

But in Notebook 1 we never wrote a gradient. We fit residuals. So here is the question: **where is the
gradient?** The answer connects the two ideas and gives the method its name.

## The loss as a function of the predictions

Write the training loss as

$$L(F) = \tfrac{1}{2}\sum_{i=1}^{n}\bigl(y_i - F_i\bigr)^2,$$

where `F_i = F(x_i)` is the model's prediction at the i-th training point. Here is the shift in
viewpoint: treat the **n predictions** `F = (F_1, …, F_n)` as the variables. Then `L` is a function of a
point in n-dimensional space — a bowl, lowest (zero) when every `F_i = y_i`. Training is walking that
point downhill toward `y`.

In Chapter 03 the variables were the weights; here they are the predictions themselves. That is the only
change — and it is why this is gradient descent in **function** space.

In [ ]:
# Treat the loss as a function of the n predictions: L(F) = 0.5 * sum((y - F)^2).
# Compute the negative gradient of L w.r.t. each prediction, numerically (finite differences),
# at the model after 3 rounds, and compare it to the residual y - F.
F3 = stages[3]
h = 1e-6
neg_grad = np.array([
    -(0.5 * (y[i] - (F3[i] + h)) ** 2 - 0.5 * (y[i] - (F3[i] - h)) ** 2) / (2 * h)
    for i in range(n)
])
residual3 = y - F3
print(f"max |(-dL/dF) - (y - F)|  =  {np.max(np.abs(neg_grad - residual3)):.2e}")
print(f"point 0:  y={y[0]:.3f}  F={F3[0]:.3f}   residual = {residual3[0]:.4f}   "
      f"-dL/dF = {neg_grad[0]:.4f}")

**Read the result.** The negative gradient of `L` with respect to each prediction is

$$-\frac{\partial L}{\partial F_i} = y_i - F_i,$$

which is exactly the **residual** — matched here to under `1e-10` (the finite-difference error; the
identity is exact). So "fit a tree to the residual" (Notebook 1) and "fit a tree to the negative
gradient" are the *same instruction*. Each round takes the current predictions one step downhill on `L`.
Because the variables are the function's values at the data, this is **gradient descent in function
space**.

## An approximate, tree-shaped step

Plain gradient descent would move each prediction on its own: `F_i ← F_i + ν·(y_i − F_i)`. Boosting
cannot do that — it moves by adding a **tree**, and a tree gives every point in a leaf the *same* value.
So the tree is a **piecewise-constant approximation** of the negative gradient: it captures the broad "up
here, down there" of the gradient, not its exact per-point value. Boosting takes approximate downhill
steps, each shaped like a tree, each scaled by `ν`.

In [ ]:
# At round 10 the negative gradient is the residual y - F; the next tree approximates it.
round_k = 10
Fk = stages[round_k]
g_k = y - Fk                  # the negative gradient at this round
step_tree = trees[round_k]    # the tree fit to this round's gradient
order = np.argsort(x)

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(x, g_k, color=COLORS["error"], edgecolor=COLORS["text"], linewidth=0.5, alpha=0.85,
           label=f"negative gradient at round {round_k}  (= y − F)")
ax.plot(x[order], step_tree.predict(X)[order], color=COLORS["model"], linewidth=2.6,
        label="the next tree (its approximation)")
ax.axhline(0.0, color=COLORS["muted"], linewidth=1.0, linestyle=":")
ax.set_xlabel("x")
ax.set_ylabel("negative gradient")
ax.set_title("each tree approximates the downhill direction")
ax.legend()
plt.show()

**Read the figure.** The coral points are the negative gradient at round 10 — the residual
`y − F`, telling each point which way and how far to move. The blue line is the next tree's prediction: a
few flat pieces that approximate that gradient (the best a depth-2 tree can manage). Boosting adds `ν`
times this step and moves on. The tree is the *direction* of the step; `ν` is its *length*.

## Watch it descend

If each round is a downhill step, the **total loss** `L = ½ Σ (y − F)²` should fall every round. (Note
`L = (n/2)·MSE`, so this is Notebook 1's MSE curve in different units.) We cannot picture all n dimensions
at once — but we can take a **2-D slice**: pick two training points and watch their predictions move while
the loss over that pair forms a simple bowl.

In [ ]:
fig, (axL, axR) = plt.subplots(1, 2, figsize=(13, 5))

# Left: a 2-D slice of function space. Pick the points nearest the sine's peak and trough.
i = int(np.argmin(np.abs(x - np.pi / 2)))      # near the peak
j = int(np.argmin(np.abs(x - 3 * np.pi / 2)))  # near the trough
Fi = np.array([s[i] for s in stages])
Fj = np.array([s[j] for s in stages])

# Restricted to (F_i, F_j), the loss is 0.5(y_i - a)^2 + 0.5(y_j - b)^2 (other terms constant):
# circular contours centred at (y_i, y_j).
a = np.linspace(min(Fi.min(), y[i]) - 0.4, max(Fi.max(), y[i]) + 0.4, 200)
b = np.linspace(min(Fj.min(), y[j]) - 0.4, max(Fj.max(), y[j]) + 0.4, 200)
A, B = np.meshgrid(a, b)
loss_slice = 0.5 * (y[i] - A) ** 2 + 0.5 * (y[j] - B) ** 2
axL.contour(A, B, loss_slice, levels=12, colors=COLORS["muted"], linewidths=1.0)
axL.plot(Fi, Fj, color=COLORS["model"], linewidth=1.8, marker="o", markersize=3,
         label="boosting trajectory")
axL.scatter([Fi[0]], [Fj[0]], color=COLORS["train"], s=90, zorder=5, edgecolor=COLORS["text"],
            label="start (the mean)")
axL.scatter([y[i]], [y[j]], color=COLORS["highlight"], marker="*", s=280, zorder=5,
            edgecolor=COLORS["text"], label="minimum (y_i, y_j)")
axL.set_xlabel("prediction at the peak point")
axL.set_ylabel("prediction at the trough point")
axL.set_title("descent on a 2-D slice of function space")
axL.legend(loc="upper right")

# Right: the total loss falls every round.
axR.plot(np.arange(len(loss_curve)), loss_curve, color=COLORS["model"], linewidth=2.4)
axR.set_xlabel("number of trees")
axR.set_ylabel("total loss  L = ½ Σ (y − F)²")
axR.set_title("the loss descends, one tree per step")
plt.show()

**Read the figure.** *Left:* a slice of function space through two predictions — the point nearest
the sine's peak and the one nearest its trough. The grey contours are the loss over that pair, a bowl
centred at the target `(y_i, y_j)` (the star). Both predictions start at the mean (the dot) and the
boosting trajectory walks them down into the bowl's minimum — gradient descent, made visible (it may
even ease slightly past the target on this pair before settling, since each tree is fit to all the points
at once, not to these two). *Right:*
the total loss across all n points falls monotonically, one step per tree, from about 30 to under 1. The
path is not a perfectly straight plunge, because each step is a tree (a shared value per leaf), not a free
per-point move — so it descends in many small, approximate steps.

## A different loss, a different gradient

The clean identity "residual = negative gradient" belongs to the **squared-error** loss. The power of
gradient boosting is that we are free to choose a different loss — and the negative gradient changes with
it. What, for instance, is the negative gradient of the **absolute-error** loss `|y − F|`?

In [ ]:
# A different loss has a different negative gradient — the "pseudo-residual" you fit.
sq_resid = y - F3                  # squared error:   -dL/dF = y - F
abs_grad = np.sign(y - F3)         # absolute error:  -d|y-F|/dF = sign(y - F)
table = pd.DataFrame({
    "y": y[:6].round(3),
    "F": F3[:6].round(3),
    "squared-error (y - F)": sq_resid[:6].round(3),
    "absolute-error sign(y - F)": abs_grad[:6],
})
print(table.to_string(index=False))
print(f"\nsquared-error pseudo-residual range:  [{sq_resid.min():.3f}, {sq_resid.max():.3f}]")
print(f"absolute-error pseudo-residual values: {np.unique(abs_grad)}")

**Read the result.** For squared error, each tree fits `y − F`: large where the miss is large, so
a single far-off point pulls hard (sensitive to outliers). For absolute error, each tree fits only the
**sign** of the miss, `±1`: every point pulls equally hard regardless of distance (robust to outliers).
Two losses, two different things to fit — yet the same loop. In Notebook 3 the loss will be log-loss and
the negative gradient will be `y − p`, which is how gradient boosting does **classification**. The
general recipe is one line: **fit the negative gradient of whatever loss you choose.** That is the
"gradient" in gradient boosting, and why the family generalizes AdaBoost's single fixed loss.

In [ ]:
# "Fit the negative gradient" is the same loop as Notebook 1 — so it still matches the library.
gbr = GradientBoostingRegressor(
    n_estimators=N_TREES, learning_rate=NU, max_depth=MAX_DEPTH,
    loss="squared_error", subsample=1.0, random_state=SEED,
).fit(X, y)
print(f"max |by-hand (fit the negative gradient) - GradientBoostingRegressor|  =  "
      f"{np.max(np.abs(F - gbr.predict(X))):.2e}")

**The name, earned.** Boosting is gradient descent in function space: each tree is an approximate
step along the negative gradient of a loss, scaled by the learning rate. scikit-learn's
`GradientBoostingRegressor` does exactly this, fitting the negative gradient of its `loss` argument every
round — which is why re-labelling our loop "fit the negative gradient" leaves the arithmetic, and the
parity, untouched. Notebook 3 swaps the loss to log-loss and the same machine starts doing
classification.

## Your turn

1. **Loss and MSE.** Confirm from the code that `L = (n/2)·MSE`, and describe in one sentence what the
   loss-vs-trees curve shows about each step.
2. **Robustness.** Move one training point far from the curve (a deliberate outlier). Compute both
   pseudo-residuals at that point — `y − F` (squared error) and `sign(y − F)` (absolute error) — and say
   which loss chases the outlier harder, and why absolute error is called *robust*.
3. **Why not exact steepest descent?** On the 2-D bowl, the trajectory does not point straight at the
   minimum each step. Explain why (the tree assigns one value per leaf, so it cannot move two points in a
   leaf independently), and what that approximation buys us in return.

## What you built

- You wrote the loss as a function of the **predictions** and showed its negative gradient is the
  **residual** `y − F` (to machine precision).
- You re-read Notebook 1's loop as **gradient descent in function space** — each tree an approximate
  downhill step, `ν` the step length.
- You watched the total loss descend, and saw the path reach the minimum on a 2-D slice.
- You saw that a **different loss** gives a **different gradient** to fit — the door to classification
  (Notebook 3) and to robust losses.

**Vocabulary you now own:** negative gradient · pseudo-residual · function space · gradient-descent step ·
loss bowl.

## Going further (optional)

- **The functional-gradient view (Friedman, 2001).** Treat the model as a *function* `F`. The loss has a
  gradient that is itself a function — its value at each point is `∂L/∂F(x_i)`. Boosting fits a tree to
  the negative of that (a **projection** of the gradient onto the space of trees), then takes a step. The
  per-leaf value is a small 1-D minimization (a *line search*): for squared error it works out to the
  mean of the residuals in the leaf — exactly what a regression tree already returns, so Notebook 1
  needed no extra step.
- **A look ahead.** Notebook 3 meets a loss whose leaf line search needs real work: classification's
  log-loss requires a one-step Newton correction in each leaf. Same machine, one honest extra step.

## References

- Friedman, J. H. (2001). *Greedy function approximation: a gradient boosting machine.* Annals of
  Statistics, 29(5), 1189–1232. DOI [10.1214/aos/1013203451](https://doi.org/10.1214/aos/1013203451).
- Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning* (2nd ed.),
  §10.10. DOI [10.1007/978-0-387-84858-7](https://doi.org/10.1007/978-0-387-84858-7).
- Gradient descent in parameter space: Chapter 03, Notebook 4 (logistic regression).

*Previous: 01 — Boosting as fitting residuals.*  ·  *Next: 03 — Gradient boosting for classification.*